# 07. Model lifecycle: versioning, drift, retraining

## Business use

A shipping model that was calibrated last quarter can go wrong in two different ways. The mix of parcels can change (heavier freight), or the world can change while the inputs look the same (a carrier raises tariffs). The first is input drift. The second is concept drift, and it shows up as miscalibration: actual costs stop landing inside the 90% intervals about 90% of the time.

## What we store per version

| Stage | What | Bayesian specifics |
|---|---|---|
| Training | Seed, data hash, code revision | Priors and the model spec saved with the run |
| Validation | Holdout and calibration | Gates: R-hat below 1.01, ESS above 400, zero divergences, posterior predictive check |
| Versioning | One registry entry | Full InferenceData for audit, thinned draws for serving |
| Monitoring | Inputs, predictions, business outcomes | Predictive-interval coverage, per segment |
| Retraining | Schedule plus a trigger | Promote only if the gates pass and a holdout beats the current version |

`model_version` is on every response and on `/health`. A minor bump is a retrain on new data. A major bump is a contract or structure change. The registry entry holds the netcdf, the npz, the metadata, the git SHA and the data hash.

Do not feed yesterday's posterior in as today's prior unless you mean to. That double-counts the overlapping data.

$$
\mathrm{PSI} = \sum_{\mathrm{bins}} (\mathrm{new}_i - \mathrm{ref}_i) \ln \frac{\mathrm{new}_i}{\mathrm{ref}_i}
$$

Rule of thumb: below 0.1 ok, 0.1 to 0.25 watch, above 0.25 alert. PSI is a reason to look. Coverage is the reason to retrain.


## Three simulated weeks

Uses the artifact from notebook 06 (`deploy/posterior_draws.npz`, `model_meta.json`, `train.csv`). The scenarios are: a week drawn from the same process, a week of heavier parcels, and a week where carrier D's tariff is 15% higher while the inputs are unchanged.


In [1]:
import importlib
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "deploy"))
drift_check = importlib.import_module("drift_check")
train = importlib.import_module("train")

if not (ROOT / "deploy" / "model_meta.json").exists():
    train.main()

rows = drift_check.main()
for row in rows:
    assert 0 <= row["coverage"] <= 1
    assert row["psi_log_weight"] >= 0

week_ok: PSI(log weight)=0.039 -> ok; 90% interval coverage on actual costs=0.909
week_heavier_parcels: PSI(log weight)=0.537 -> ALERT; 90% interval coverage on actual costs=0.885
carrier_d +15% tariff: PSI(log weight)=0.014; coverage all=0.871; coverage carrier_d=0.849


## How to read the outputs

| Scenario | What you should see | What to do |
|---|---|---|
| Ordinary week | PSI below 0.1, coverage near 0.90 | Nothing |
| Heavier parcels | PSI above 0.25 | Look. Coverage can stay near 0.90 if the weight slope still extrapolates. Input drift is not an automatic retrain. |
| Carrier D +15% tariff | PSI near 0, because weights did not change | Coverage for that carrier falls. Input monitoring is blind to it. Retrain, or add a tariff feature. |

Watch coverage per carrier, not only overall. A single carrier can be miscalibrated while the average still looks fine.

A reference run of these three weeks reported PSI about 0.04 (ok) and coverage about 0.91; PSI about 0.54 (alert) and coverage about 0.89 on heavier parcels; and a tariff shock with PSI about 0.01, overall coverage about 0.87, and carrier D coverage about 0.85. The thresholds, not the third digit, are the decision.

## Saying uncertainty in the language of a decision

| Technical claim | What to say |
|---|---|
| 90% predictive interval from 6 to 14 USD | In 9 of 10 shipments like this, the cost lands between 6 and 14 USD. |
| $P(\text{revenue up}) = 0.40$ | Raising this price is close to a coin flip. Test it before a rollout. |
| Coverage 0.85 on one carrier | The intervals for that carrier are now too narrow or shifted. We retrain. |
| R-hat 1.00, ESS in the thousands | Rerunning the fit would not change the quote we publish. |
| Prior Normal(-1, 0.5) on elasticity | We assume demand falls when price rises. The data can override how much. |
| A wide interval | We know less here. This is where more data or an experiment pays off. |

Lead with the decision, then the range in frequency terms ("9 out of 10"), then the assumption that would change the call. Leave trace plots out of the meeting. A credible interval is a probability statement about the parameter given the model. A confidence interval is not. Do not describe one as the other.

## Pitfalls

- Paging someone on PSI alone. It is noisy, and it cannot see a tariff change.
- A scheduled retrain with no gates, which can ship a worse model.
- A version with no prior and no data hash, which cannot be reproduced.
- Using yesterday's posterior as today's prior and double-counting the data.
- One coverage number for every carrier.


## How to explain this to a CTO


In [2]:
by_name = {row["name"]: row for row in rows}
heavy = by_name["week_heavier_parcels"]
tariff = by_name["carrier_d_tariff"]
print(
    "We watch two things: whether this week's parcels look like the ones the model learned from, "
    "and whether real costs still land inside our 90% ranges about 90% of the time."
)
print(
    f"Heavier parcels move the input score to {heavy['psi_log_weight']:.2f} "
    f"({heavy['status']}). Coverage is still {heavy['coverage']:.0%}. "
    "That is a reason to look at the mix, not an automatic retrain, as long as the ranges still cover."
)
print(
    f"A 15% tariff on one carrier leaves the input score at {tariff['psi_log_weight']:.2f} "
    f"({tariff['status']}), so input monitoring stays quiet, "
    f"but that carrier's coverage falls to {tariff['coverage_carrier_d']:.0%}. "
    "Coverage is what triggers a retrain. A new version ships only if it passes the same automated checks "
    "and beats the current one on a holdout."
)

We watch two things: whether this week's parcels look like the ones the model learned from, and whether real costs still land inside our 90% ranges about 90% of the time.
Heavier parcels move the input score to 0.54 (ALERT). Coverage is still 88%. That is a reason to look at the mix, not an automatic retrain, as long as the ranges still cover.
A 15% tariff on one carrier leaves the input score at 0.01 (ok), so input monitoring stays quiet, but that carrier's coverage falls to 85%. Coverage is what triggers a retrain. A new version ships only if it passes the same automated checks and beats the current one on a holdout.


## Check yourself

**How do you decide to retrain?**
On a cadence, and earlier when segment-level 90% coverage falls to about 0.85. Promote the candidate only if R-hat, ESS and divergences pass and the holdout beats the incumbent. Keep the previous version so you can roll back.

**How do you version the model?**
Semantic version in every response. The registry keeps the audit netcdf, the serving draws, the code revision, the training-data hash, the priors, and the diagnostics. A contract change is a new path, not only a new number.
